# Workshop 3 — Word embeddings (QUAERO)

**Objective.** Build and compare word embeddings with `gensim` and `fasttext`, trained
on two corpora, and evaluate them with cosine similarity. The models are reused for
the NER task in the next session.

| | |
|---|---|
| Corpora | `QUAERO_FrenchMed` (small, medical), `QUAERO_FrenchPress` (large, non-medical) |
| Approaches | word2vec CBOW, word2vec skip-gram, fastText CBOW |
| Models | 2 corpora x 3 approaches = **6 models** |
| Hyper-parameters | `dim = 100`, `min_count = 1` |


## 0. Requirements


In [1]:
# on Google Colab, run this first
# %pip install gensim fasttext


In [2]:
from pathlib import Path

import pandas as pd
from gensim.models import Word2Vec
from gensim.models.fasttext import load_facebook_vectors
import fasttext

TP = Path('TP_ISD2020')

# the two corpora: tokenized text, one sentence per line
CORPORA = {
    'med':   TP / 'QUAERO_FrenchMed'   / 'QUAERO_FrenchMed_traindev.ospl',    # small, medical
    'press': TP / 'QUAERO_FrenchPress' / 'QUAERO_FrenchPress_traindev.ospl',  # large, non-medical
}

APPROACHES = ['w2v_cbow', 'w2v_skipgram', 'ft_cbow']

# hyper-parameters given in the lab description
DIM = 100        # size of a word vector
MIN_COUNT = 1    # keep every word, even those seen once

WORKERS = 4      # threads
SEED = 42

# where the models and the vectors are saved
MODELS = Path('models')
MODELS.mkdir(exist_ok=True)


## 1. The two corpora

The files are already tokenized, so splitting on spaces is enough.


In [3]:
def read_sentences(path):
    """Read an .ospl file as a list of sentences, each sentence a list of tokens."""
    lines = path.read_text(encoding='utf-8').splitlines()
    return [tokens for line in lines if (tokens := line.split())]


sentences = {name: read_sentences(path) for name, path in CORPORA.items()}

stats = pd.DataFrame({
    name: {
        'sentences': len(s),
        'tokens': sum(len(x) for x in s),
        'word types': len({w for x in s for w in x}),
    }
    for name, s in sentences.items()
})
display(stats)

print('med   :', ' '.join(sentences['med'][12]))
print('press :', ' '.join(sentences['press'][0]))


,med,press
sentences,3021,38548
tokens,51896,1251561
word types,9104,39654


med   : La plupart des patients nécessitent des doses inférieures à 9 , 6 microgrammes par jour .
press : Patricia Martin , que voici , que voilà ! oh , bonjour Nicolas Stoufflet .


The press corpus is about **24x larger** than the medical one. This difference in
quantity is part of what the second evaluation measures, along with the difference in
domain.


## 2. Training the 6 models

One function per approach. Only `dim` and `min_count` are imposed by the lab, so the
other parameters keep their library default (window 5, 5 epochs), which is the same
for gensim and fastText and therefore keeps the comparison fair.


### 2.1 word2vec — CBOW and skip-gram (gensim)

`sg=0` is CBOW (predict the word from its context), `sg=1` is skip-gram
(predict the context from the word).


In [4]:
def train_word2vec(corpus_name, sg):
    """Train a word2vec model and save it. sg=0 -> CBOW, sg=1 -> skip-gram."""
    model = Word2Vec(
        sentences=sentences[corpus_name],
        vector_size=DIM,
        min_count=MIN_COUNT,
        sg=sg,
        workers=WORKERS,
        seed=SEED,
    )
    name = 'w2v_skipgram' if sg else 'w2v_cbow'
    model.save(str(MODELS / f'{corpus_name}_{name}.model'))
    return model.wv


### 2.2 fastText — CBOW (fasttext)

fastText reads the corpus file directly. A word vector is the sum of the vectors of
its character n-grams, so fastText can also build a vector for an unknown word.
The saved `.bin` is read back with gensim, so the 6 models end up with the same
`KeyedVectors` interface.


In [5]:
def train_fasttext(corpus_name):
    """Train a fastText CBOW model, save it, and return it as gensim vectors."""
    model = fasttext.train_unsupervised(
        str(CORPORA[corpus_name]),
        model='cbow',
        dim=DIM,
        minCount=MIN_COUNT,
        thread=WORKERS,
        bucket=200_000,   # subword hash table (the 2M default is oversized for these corpora)
        verbose=0,        # hide the training progress bar
    )
    path = MODELS / f'{corpus_name}_ft_cbow.bin'
    model.save_model(str(path))
    return load_facebook_vectors(str(path))


### 2.3 Train and save the 6 models


In [6]:
# emb['med']['w2v_cbow'] -> the vectors of that corpus / approach pair
emb = {}

for corpus_name in CORPORA:
    emb[corpus_name] = {
        'w2v_cbow':     train_word2vec(corpus_name, sg=0),
        'w2v_skipgram': train_word2vec(corpus_name, sg=1),
        'ft_cbow':      train_fasttext(corpus_name),
    }
    print(f'{corpus_name}: 3 models trained')

display(pd.DataFrame({c: {a: len(emb[c][a]) for a in APPROACHES} for c in CORPORA}))


med: 3 models trained
press: 3 models trained


,med,press
w2v_cbow,9104,39654
w2v_skipgram,9104,39654
ft_cbow,9105,39655


With `min_count = 1` the vocabulary of each model is the full vocabulary of its
corpus, so every candidate word below is guaranteed to be present.


### 2.4 Save the word embeddings

Besides the models, the vectors are exported in the plain-text word2vec format
(`.vec`), which is what a `scipy`-based evaluation would load.


In [7]:
for corpus_name in CORPORA:
    for approach in APPROACHES:
        emb[corpus_name][approach].save_word2vec_format(
            str(MODELS / f'{corpus_name}_{approach}.vec')
        )

for f in sorted(MODELS.iterdir()):
    print(f'{f.name:32s} {f.stat().st_size / 1e6:7.1f} MB')


med_ft_cbow.bin                     87.5 MB
med_ft_cbow.vec                     10.5 MB
med_w2v_cbow.model                   7.6 MB
med_w2v_cbow.vec                    11.7 MB
med_w2v_skipgram.model               7.6 MB
med_w2v_skipgram.vec                11.3 MB
press_ft_cbow.bin                  112.5 MB
press_ft_cbow.vec                   45.4 MB
press_w2v_cbow.model                33.0 MB
press_w2v_cbow.vec                  49.6 MB
press_w2v_skipgram.model            33.0 MB
press_w2v_skipgram.vec              47.7 MB


## 3. Semantic similarity

For each candidate word we look at the 10 closest words by cosine similarity, using
gensim's `most_similar`. The corpora are in French, so the candidate list is used in
its French form.


In [8]:
WORDS = {
    'patient': 'patient',
    'treatment': 'traitement',
    'disease': 'maladie',
    'solution': 'solution',
    'yellow': 'jaune',
}

TOPN = 10


def closest(kv, word, topn=TOPN):
    """The topn closest words by cosine similarity, as 'word (score)' strings."""
    if word not in kv:
        return ['(not in vocabulary)'] + [''] * (topn - 1)
    return [f'{w} ({s:.2f})' for w, s in kv.most_similar(word, topn=topn)]


def table(columns, word):
    """Build a rank x columns table of closest words. columns is {label: vectors}."""
    df = pd.DataFrame({label: closest(kv, word) for label, kv in columns.items()})
    df.index = range(1, len(df) + 1)
    df.index.name = word
    return df


# how often each candidate word occurs in each corpus
counts = pd.DataFrame({
    c: {fr: emb[c]['w2v_cbow'].get_vecattr(fr, 'count') for fr in WORDS.values()}
    for c in CORPORA
})
display(counts)


,med,press
patient,33,11
traitement,251,53
maladie,65,114
solution,67,100
jaune,9,23


### 3.1 Evaluation 1 — same corpus, different approaches

The corpus is fixed, only the algorithm changes. This isolates the impact of the
**approach**.


#### On the medical corpus


In [9]:
for fr in WORDS.values():
    display(table({a: emb['med'][a] for a in APPROACHES}, fr))


,w2v_cbow,w2v_skipgram,ft_cbow
patient,,,
1,sont (1.00),symptômes (1.00),patiente (1.00)
2,les (1.00),qui (1.00),Patient (1.00)
3,en (1.00),risque (1.00),patients (1.00)
4,le (1.00),Tysabri (1.00),conditionnement (1.00)
5,et (1.00),réactions (1.00),entrainement (1.00)
6,que (1.00),nécessaire (1.00),contient (1.00)
7,par (1.00),devra (0.99),appartient (1.00)
8,est (1.00),qu (0.99),conscient (1.00)
9,Le (1.00),arrêt (0.99),préalablement (1.00)


,w2v_cbow,w2v_skipgram,ft_cbow
traitement,,,
1,des (1.00),TYSABRI (0.98),Traitements (1.00)
2,Le (1.00),médicaments (0.97),traitements (1.00)
3,en (1.00),pendant (0.97),Traitement (1.00)
4,dans (1.00),le (0.97),Taaitement (1.00)
5,avec (1.00),par (0.97),Allaitement (1.00)
6,les (1.00),Le (0.97),étroitement (1.00)
7,par (1.00),association (0.97),typiquement (1.00)
8,La (1.00),avec (0.96),retentissement (1.00)
9,après (1.00),infection (0.96),spécialement (1.00)


,w2v_cbow,w2v_skipgram,ft_cbow
maladie,,,
1,de (1.00),enfant (0.99),Maladie (1.00)
2,la (1.00),étude (0.99),maladies (1.00)
3,du (1.00),adulte (0.99),malades (1.00)
4,dans (1.00),un (0.99),contraception (1.00)
5,. (1.00),infection (0.98),préalablement (1.00)
6,des (1.00),après (0.98),recrutement (1.00)
7,et (1.00),cancer (0.98),reconstitution (1.00)
8,à (1.00),administration (0.98),électroencéphalographiques (1.00)
9,en (1.00),une (0.98),conjointement (1.00)


,w2v_cbow,w2v_skipgram,ft_cbow
solution,,,
1,mg (1.00),flacon (0.98),évolution (1.00)
2,perfusion (1.00),contient (0.98),Dissolution (1.00)
3,pour (1.00),100 (0.98),consommation (1.00)
4,) (1.00),jour (0.97),Evolution (1.00)
5,( (1.00),fois (0.97),Solution (1.00)
6,ou (1.00),mg (0.97),solutions (1.00)
7,flacon (1.00),300 (0.97),Pollution (1.00)
8,2 (1.00),150 (0.97),evolution (1.00)
9,1 (1.00),perfusion (0.97),section (1.00)


,w2v_cbow,w2v_skipgram,ft_cbow
jaune,,,
1,) (1.00),23 (1.00),parisienne (1.00)
2,( (1.00),14 (1.00),spéciale (1.00)
3,. (1.00),24 (1.00),rabique (1.00)
4,: (1.00),Après (1.00),embolique (1.00)
5,3 (1.00),34 (1.00),confusionnel (1.00)
6,", (1.00)",rhabdomyolyse (1.00),urique (1.00)
7,• (1.00),18 (1.00),Département (1.00)
8,dans (1.00),France (1.00),hémorragique (1.00)
9,à (1.00),Fluoxétine (1.00),serique (1.00)


#### On the press corpus


In [10]:
for fr in WORDS.values():
    display(table({a: emb['press'][a] for a in APPROACHES}, fr))


,w2v_cbow,w2v_skipgram,ft_cbow
patient,,,
1,parcours (0.88),qualitatif (0.95),impatient (0.99)
2,consensus (0.84),regret (0.95),patientent (0.98)
3,défi (0.83),diplôme (0.95),échoient (0.98)
4,malaise (0.83),colis (0.94),soutient (0.98)
5,document (0.82),identification (0.94),ratifient (0.98)
6,désir (0.82),étudiant (0.94),abstient (0.97)
7,système (0.82),truc (0.94),contient (0.97)
8,mythe (0.81),mollah (0.94),engagent (0.97)
9,cri (0.81),dissuasives (0.94),refusent (0.97)


,w2v_cbow,w2v_skipgram,ft_cbow
traitement,,,
1,calendrier (0.92),juridique (0.89),concrètement (0.98)
2,employer (0.92),potentiel (0.89),enfermement (0.98)
3,parc (0.92),fondamental (0.89),promptement (0.97)
4,vin (0.91),coût (0.89),probablement (0.97)
5,laboratoire (0.91),mépris (0.89),logement (0.97)
6,fardeau (0.91),financement (0.89),rapidement (0.97)
7,tabac (0.91),pacte (0.88),enlisement (0.97)
8,café (0.91),changement (0.88),difficilement (0.97)
9,décor (0.91),outil (0.88),contrairement (0.97)


,w2v_cbow,w2v_skipgram,ft_cbow
maladie,,,
1,polémique (0.93),totale (0.86),malnutrie (0.95)
2,procédure (0.93),supplémentaire (0.84),malade (0.93)
3,négociation (0.92),garantie (0.83),plaie (0.93)
4,dictature (0.92),trace (0.83),vivace (0.93)
5,zone (0.92),misère (0.82),persifle (0.92)
6,population (0.91),dictature (0.82),irréversible (0.92)
7,puissance (0.91),nationalité (0.82),calcule (0.92)
8,règle (0.91),commande (0.82),faiblesse (0.92)
9,bataille (0.91),liée (0.82),faible (0.92)


,w2v_cbow,w2v_skipgram,ft_cbow
solution,,,
1,garantie (0.93),pacifique (0.89),résolution (0.99)
2,catastrophe (0.91),alternative (0.88),révolution (0.99)
3,forme (0.91),démocratie (0.87),dissolution (0.99)
4,seule (0.91),règle (0.87),évolution (0.99)
5,meilleure (0.90),définition (0.86),dilution (0.99)
6,notion (0.90),indispensable (0.85),pollution (0.99)
7,règle (0.90),difficulté (0.85),caution (0.99)
8,valeur (0.90),puissance (0.85),fonction (0.98)
9,censure (0.89),consensuelle (0.85),imposition (0.98)


,w2v_cbow,w2v_skipgram,ft_cbow
jaune,,,
1,maillot (0.96),maillot (0.93),brune (0.97)
2,verbe (0.96),lauréat (0.89),l'une (0.97)
3,libéral (0.95),emparé (0.88),d'une (0.97)
4,Perrot (0.95),Ilton (0.87),lagune (0.96)
5,constructeur (0.95),sprint (0.87),lune (0.96)
6,hamburger (0.95),Cooke (0.87),Bjarne (0.95)
7,sévère (0.95),cavalier (0.87),Jeune (0.95)
8,LVMH (0.95),trophée (0.87),orne (0.95)
9,colombien (0.95),lion (0.87),one (0.95)


### 3.2 Evaluation 2 — same approach, different corpora

The approach is fixed, only the training data changes. This isolates the impact of
the **data**, both its type (medical vs general) and its quantity (52K vs 1.25M tokens).


In [11]:
for approach in APPROACHES:
    print(f'=== {approach} ===')
    for fr in WORDS.values():
        display(table({c: emb[c][approach] for c in CORPORA}, fr))


=== w2v_cbow ===


,med,press
patient,,
1,sont (1.00),parcours (0.88)
2,les (1.00),consensus (0.84)
3,en (1.00),défi (0.83)
4,le (1.00),malaise (0.83)
5,et (1.00),document (0.82)
6,que (1.00),désir (0.82)
7,par (1.00),système (0.82)
8,est (1.00),mythe (0.81)
9,Le (1.00),cri (0.81)


,med,press
traitement,,
1,des (1.00),calendrier (0.92)
2,Le (1.00),employer (0.92)
3,en (1.00),parc (0.92)
4,dans (1.00),vin (0.91)
5,avec (1.00),laboratoire (0.91)
6,les (1.00),fardeau (0.91)
7,par (1.00),tabac (0.91)
8,La (1.00),café (0.91)
9,après (1.00),décor (0.91)


,med,press
maladie,,
1,de (1.00),polémique (0.93)
2,la (1.00),procédure (0.93)
3,du (1.00),négociation (0.92)
4,dans (1.00),dictature (0.92)
5,. (1.00),zone (0.92)
6,des (1.00),population (0.91)
7,et (1.00),puissance (0.91)
8,à (1.00),règle (0.91)
9,en (1.00),bataille (0.91)


,med,press
solution,,
1,mg (1.00),garantie (0.93)
2,perfusion (1.00),catastrophe (0.91)
3,pour (1.00),forme (0.91)
4,) (1.00),seule (0.91)
5,( (1.00),meilleure (0.90)
6,ou (1.00),notion (0.90)
7,flacon (1.00),règle (0.90)
8,2 (1.00),valeur (0.90)
9,1 (1.00),censure (0.89)


...

,med,press
patient,,
1,patiente (1.00),impatient (0.99)
2,Patient (1.00),patientent (0.98)
3,patients (1.00),échoient (0.98)
4,conditionnement (1.00),soutient (0.98)
5,entrainement (1.00),ratifient (0.98)
6,contient (1.00),abstient (0.97)
7,appartient (1.00),contient (0.97)
8,conscient (1.00),engagent (0.97)
9,préalablement (1.00),refusent (0.97)


,med,press
traitement,,
1,Traitements (1.00),concrètement (0.98)
2,traitements (1.00),enfermement (0.98)
3,Traitement (1.00),promptement (0.97)
4,Taaitement (1.00),probablement (0.97)
5,Allaitement (1.00),logement (0.97)
6,étroitement (1.00),rapidement (0.97)
7,typiquement (1.00),enlisement (0.97)
8,retentissement (1.00),difficilement (0.97)
9,spécialement (1.00),contrairement (0.97)


,med,press
maladie,,
1,Maladie (1.00),malnutrie (0.95)
2,maladies (1.00),malade (0.93)
3,malades (1.00),plaie (0.93)
4,contraception (1.00),vivace (0.93)
5,préalablement (1.00),persifle (0.92)
6,recrutement (1.00),irréversible (0.92)
7,reconstitution (1.00),calcule (0.92)
8,électroencéphalographiques (1.00),faiblesse (0.92)
9,conjointement (1.00),faible (0.92)


,med,press
solution,,
1,évolution (1.00),résolution (0.99)
2,Dissolution (1.00),révolution (0.99)
3,consommation (1.00),dissolution (0.99)
4,Evolution (1.00),évolution (0.99)
5,Solution (1.00),dilution (0.99)
6,solutions (1.00),pollution (0.99)
7,Pollution (1.00),caution (0.99)
8,evolution (1.00),fonction (0.98)
9,section (1.00),imposition (0.98)


,med,press
jaune,,
1,parisienne (1.00),brune (0.97)
2,spéciale (1.00),l'une (0.97)
3,rabique (1.00),d'une (0.97)
4,embolique (1.00),lagune (0.96)
5,confusionnel (1.00),lune (0.96)
6,urique (1.00),Bjarne (0.95)
7,Département (1.00),Jeune (0.95)
8,hémorragique (1.00),orne (0.95)
9,serique (1.00),one (0.95)


### 3.3 Cross-check with scipy

`most_similar` ranks by cosine similarity. We verify this on one pair by computing
the same value with `scipy.spatial`
(cosine similarity = 1 - cosine distance).


In [12]:
from scipy.spatial.distance import cosine

kv = emb['med']['w2v_skipgram']
word = 'maladie'
neighbour, gensim_score = kv.most_similar(word, topn=1)[0]
scipy_score = 1 - cosine(kv[word], kv[neighbour])

print(f'{word} / {neighbour}')
print(f'  gensim most_similar : {gensim_score:.6f}')
print(f'  1 - scipy cosine    : {scipy_score:.6f}')


maladie / enfant
  gensim most_similar : 0.988822
  1 - scipy cosine    : 0.988822


---

## 4. Findings

### 4.1 Impact of the approach (same corpus)

**CBOW vs skip-gram.** On the press corpus skip-gram is clearly the better of the two.
For *jaune* it returns maillot, trophee, chelem, sprint, demi-finaliste — the whole
*maillot jaune* / sports field — while CBOW mixes unrelated words (Lars, severe,
docteur, von). For *solution* skip-gram finds pacifique, consensuelle, alternative,
where CBOW returns certaine, seule, serrure. CBOW averages the whole context window
into one prediction, which blurs infrequent words; skip-gram trains on each
context word separately and keeps them better separated.

**fastText.** Its neighbours are decided by spelling, not by meaning. *solution* gives
resolution, revolution, dissolution, evolution, dilution, pollution; *traitement*
gives concretement, promptement, probablement; *jaune* gives brune, lune, lagune,
Neptune. This follows directly from building a word vector out of character n-grams.
It is a real weakness for pure semantic similarity, but the same mechanism is the
only thing that produces usable output on the small medical corpus, where it groups
*maladie* with Maladie, maladies, malades, and *traitement* with traitements,
Traitement, Traitements — and even catches the typo *Taaitement*. It also gives a
vector to any unseen word, which should help for NER.

### 4.2 Impact of the data (same approach)

**Quantity.** The medical corpus is 24x smaller (52K vs 1.25M tokens) and the two
word2vec models trained on it fail: nearly every similarity rounds to 1.00 and the
closest words are function words (*patient* -> sont, les, le, en, par; *traitement* ->
Le, avec, dans, en). The vectors never separate, so the ranking carries no
information. The same approaches on the press corpus give ranked, readable
neighbours. Quantity is the dominant factor in this comparison.

Note this is a training-budget effect as much as a data effect: 52K tokens over the
5 default epochs is only about 260K weight updates. `min_count = 1` makes it worse,
since it keeps every word seen once — 9,104 types for 52K tokens in the medical
corpus, 39,654 types for the press one.

**Type.** Where the medical models do carry signal, it is unmistakably in-domain:
the CBOW neighbours of *maladie* include urticaire, arthralgie, Fluoxetine, and the
fastText neighbours of *solution* are Dissolution, dilution — the liquid-preparation
sense used in drug leaflets. The press corpus encodes a different sense of the very
same words:

| Word | press (general) | Sense captured |
|---|---|---|
| *jaune* | maillot, trophee, chelem, sprint | *maillot jaune* — cycling, not the colour |
| *patient* | impatient, digne, innocent, honnete | the **adjective**, not the sick person |
| *solution* | pacifique, consensuelle, alternative | a political settlement, not a liquid |

**Conclusion for the NER step.** The two corpora do not merely differ in quality, they
encode different senses of the candidate words, so the large press corpus is not a
substitute for in-domain data on a medical task. But the medical corpus on its own is
too small to train usable word2vec vectors at these settings. For medical NER the
realistic options are to add the supplementary MEDLINE corpus provided with the TP,
or to rely on fastText, which degrades the most gracefully on little data.
